## Kelley hu4d5-5 VL-R66G charging step error analysis

In this notebook we will use a random forest model to find the most energetically influential degrees of freedom for the VL-R66G charging step TI production run. Next, we will compare the sampling of these DOF during TI production to a free energy profile derived from end state GaMD sampling. We will attempt to  correct any inaccurate sampling in the TI data and find the estimated ddG before and after the correction. 

In [1]:
import os
os.chdir("..")
from common_functions import *

### Ingesting original TI lambda production data

In [2]:
os.chdir("./TI_data/VL-R66G")
geom_dvdls_crg = pd.read_csv("R66G_crg_bound.csv")
geom_dvdls_crg_ub = pd.read_csv("R66G_crg_unbound.csv")
geom_dvdls_vdw = pd.read_csv("R66G_vdw_bound.csv")
geom_dvdls_vdw_ub = pd.read_csv("R66G_vdw_unbound.csv")


### Initial VL-R66G TI ddG estimate:

In [3]:
orig_vdw_dGs = np.array(bootstrap_df(geom_dvdls_vdw, 1000, 200))
orig_vdw_ub_dGs = np.array(bootstrap_df(geom_dvdls_vdw_ub, 1000, 200))
orig_crg_dGs = np.array(bootstrap_df(geom_dvdls_crg, 1000, 200))
orig_crg_ub_dGs = np.array(bootstrap_df(geom_dvdls_crg_ub, 1000, 200))

In [7]:
total_ddGs = print_summary_crg(orig_crg_dGs, orig_crg_ub_dGs, orig_vdw_dGs, orig_vdw_ub_dGs)
empirical_value = 0.22
orig_error = abs((np.mean(total_ddGs)) - empirical_value)

print(f"\nEmpirical value: {empirical_value} kcal/mol")
print(f"Original error: {round(orig_error, 4)} kcal/mol")

Charging step ddG: -0.8569 +/- 0.8902
Lower CI: -1.7064
Upper CI: 0.0334

Vdw step ddG: 0.1667 +/- 0.607
Lower CI: -0.4404
Upper CI: 0.75

Total ddG: -0.6902 +/- 1.0979
Lower CI: -1.7723
Upper CI: 0.4077

Empirical value: 0.22 kcal/mol
Original error: 0.9102 kcal/mol


### Charging step RF model

#### Splitting data into independent/dependent variables for random forest model

See our methods/supplemental methods section for our process to choose the input features.

In [8]:
X = geom_dvdls_crg.drop(
    ["weight_dvdl", "dvdl", "Run", "Lambda", "#Frame", "R66_T69", "R66_G68"
    ], axis=1)

X_scl = pd.DataFrame(StandardScaler().fit_transform(X))
X_scl.columns = X.columns
Y = geom_dvdls_crg["weight_dvdl"]


#### Checking for cross-correlation among independent variables

In [9]:
absCorr = abs(X_scl.corr())
for i in absCorr.columns:
    for j in absCorr.index:
        cor = absCorr.loc[i, j]
        if abs(cor) > 0.5 and i != j:
            print(i, j)
            print(cor)
            

#### Using random forest model to identify the most influential degrees of freedom

We run our model 25 times, then sort the results by the mean of feature importance across the 25 iterations. The model found that VL-V29 side chain rotamers chi1 and chi2 were the most influential nearby DOF on TI DV/DL.

In [10]:
rfeDefault = RFE(estimator=DecisionTreeRegressor(max_depth=5, random_state=42), n_features_to_select=0.75, step=0.05)
rfDefault = RandomForestRegressor(
    max_depth=10, n_estimators=200, oob_score=True, max_features=0.6, min_samples_leaf = 7, min_samples_split=14, random_state=42,
    n_jobs=-1
)

pipelineDefault_rf = Pipeline([
    ('feature_scaling', StandardScaler()),
    ('feature_selection', rfeDefault),
    ('regression_model', rfDefault)
])


imps = benchmark_model(pipelineDefault_rf, X_scl, Y, geom_dvdls_crg["Lambda"])
imps[["Mean", "Median"]].sort_values(by="Mean", ascending=False)[:15]

Avg. training r2: 
0.7567
Training r2 std dev: 
0.0014
Avg. test r2: 
0.6817
Testing r2 std dev: 
0.0046


,Mean,Median
V384_chi2,0.134898,0.109873
V384_chi1,0.111527,0.043097
N385_chi1,0.106661,0.103013
D383_chi1,0.089471,0.089556
T427_chi1,0.076006,0.075936
R66_N28,0.069066,0.068714
S407_chi1,0.067584,0.065869
N385_chi2,0.067436,0.062904
D425_chi2,0.052056,0.052453
S420_chi1,0.046894,0.029463


### Comparing the TI rotamers with the GaMD pmf (bound state)

The RF model tells us that the most energetically influential features to DV/DL are "V384_chi1" and "V384_chi2", aka VL-V29 side chain rotamers chi1 and chi2. We used GaMD on the WT hu4d5-5 complex to obtain enhanced sampling data for this end state. Then, we used PyReweighting-2D to generate a 2-D PMF of VH-V29 chi1 and VL-V29 chi2 to give us an idea of the ideal sampling of these two energetically influential features. Finally, we plot the VL-V29 rotamers explored during TI overlaid onto the GaMD pmf to probe the quality of sampling during TI.

Next, we will compare the proportion of TI sampling in each of the three predominant microstates to the relative areas of the microstates in the GaMD end state free energy profile.

In [12]:
os.chdir("../../gamd_pmfs/VL-R66G")
v164_pmf = get_pmf_2d(
    "./pmf-c2-V164c1c2_conv_300ns.dat.xvg"
)

geom_dvdls_crg["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_crg, "V384_chi1", "V384_chi2", v164_pmf, 10)

In [24]:
fig = plot_pmf_with_TI_2d_fig(v164_pmf, geom_dvdls_crg[geom_dvdls_crg["in_pmf_cont"]], "V384_chi1", "V384_chi2")
fig.update_xaxes(
    title="VL-V29 chi1"
).update_yaxes(
    title="VL-V29 chi2"
).update_layout(width=1000, height=700)
fig.show(renderer="svg")

### Checking sampling of the three microstates

The first microstate (leftmost) has insufficient sampling in many lambdas. We will run more TI lambda production with starting rotamers specifically in this microstate to boost sampling.

In [14]:
st1_crg = geom_dvdls_crg[
    (geom_dvdls_crg["in_pmf_cont"]) &
    (geom_dvdls_crg["V384_chi1"] > -120) &
    (geom_dvdls_crg["V384_chi1"] < -25)      
]

st2_crg = geom_dvdls_crg[
    (geom_dvdls_crg["in_pmf_cont"]) &
    (geom_dvdls_crg["V384_chi1"] > 0) &
    (geom_dvdls_crg["V384_chi1"] < 120)      
]

st3_crg = geom_dvdls_crg[
    (geom_dvdls_crg["in_pmf_cont"]) &
    (geom_dvdls_crg["V384_chi1"] > 120) |
    (geom_dvdls_crg["V384_chi1"] < -150)      
]

print(st1_crg.groupby("Lambda").count()["weight_dvdl"])
print(st2_crg.groupby("Lambda").count()["weight_dvdl"])
print(st3_crg.groupby("Lambda").count()["weight_dvdl"])

Lambda
1      13
2     178
3     158
4      86
5       5
8      70
9     350
10    332
11     69
12    112
Name: weight_dvdl, dtype: int64
Lambda
1     443
2     454
3     394
4     215
5     202
6     288
7     560
8     394
9     160
10    167
11    340
12    196
Name: weight_dvdl, dtype: int64
Lambda
1     533
2     295
3     400
4     665
5     780
6     695
7     421
8     486
9     400
10    399
11    544
12    630
Name: weight_dvdl, dtype: int64


### Ingesting boosted TI lambda production sampling on the leftmost state

In [15]:
os.chdir("../../TI_data/VL-R66G")
geom_dvdls_boost = pd.read_csv("R66G_crg_boosted.csv")

geom_dvdls_boost["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_boost, "V384_chi1", "V384_chi2", v164_pmf, 10)

#### Checking the sampling on the microstates to make sure it is sufficient

In [19]:
st1boost = geom_dvdls_boost[
    (geom_dvdls_boost["in_pmf_cont"]) &
    (geom_dvdls_boost["V384_chi1"] > -150) &
    (geom_dvdls_boost["V384_chi1"] < -25)      
]

st2boost = geom_dvdls_boost[
    (geom_dvdls_boost["in_pmf_cont"]) &
    (geom_dvdls_boost["V384_chi1"] > 0) &
    (geom_dvdls_boost["V384_chi1"] < 120)      
]

st3boost = geom_dvdls_boost[
    (geom_dvdls_boost["in_pmf_cont"]) &
    ((geom_dvdls_boost["V384_chi1"] > 120) |
    (geom_dvdls_boost["V384_chi1"] < -150))    
]

print(st1boost.groupby("Lambda").count()["weight_dvdl"])
print(st2boost.groupby("Lambda").count()["weight_dvdl"])
print(st3boost.groupby("Lambda").count()["weight_dvdl"])

st1boost_dGs = np.array(bootstrap_df(st1boost, 1000, 200))
st2boost_dGs = np.array(bootstrap_df(st2boost, 1000, 200))
st3boost_dGs = np.array(bootstrap_df(st3boost, 1000, 200))

Lambda
1     1151
2     1208
3     1634
4     1075
5      209
6      234
7       94
8      355
9      502
10     390
11     981
12     747
Name: weight_dvdl, dtype: int64
Lambda
1      643
2     1019
3      493
4      592
5     1626
6      731
7     1298
8      951
9     1794
10    1850
11     680
12     386
Name: weight_dvdl, dtype: int64
Lambda
1      829
2      415
3      432
4      977
5     1007
6     1851
7     1502
8     1541
9      484
10     580
11     892
12    1395
Name: weight_dvdl, dtype: int64


### Comparing TI sampling of the three microstates to the GaMD free energy profile areas

In [30]:
a1 = v164_pmf[(v164_pmf["X"] > -120) & (v164_pmf["X"] < -25)]["pop"].sum()
a2 = v164_pmf[(v164_pmf["X"] > 0) & (v164_pmf["X"] < 120)]["pop"].sum()
a3 = v164_pmf[(v164_pmf["X"] > 120) | (v164_pmf["X"] < -150)]["pop"].sum()

a1_prop = a1/(a1 + a2 + a3)
a2_prop = a2/(a1 + a2 + a3)
a3_prop = a3/(a1 + a2 + a3)

print("GaMD pmf left hand state proportion: ")
print(round(a1_prop, 4))
print("GaMD pmf middle state proportion: ")
print(round(a2_prop, 4))
print("GaMD pmf right hand state proportion: ")
print(round(a3_prop, 4))


ti1 = len(st1_crg)
ti2 = len(st2_crg)
ti3 = len(st3_crg)

print("TI left hand state proportion: ")
print(round(ti1/(ti1 + ti2 + ti3), 4))
print("TI middle hand state proportion: ")
print(round(ti2/(ti1 + ti2 + ti3), 4))
print("TI right hand state proportion: ")
print(round(ti3/(ti1 + ti2 + ti3), 4))


GaMD pmf left hand state proportion: 
0.0963
GaMD pmf middle state proportion: 
0.2731
GaMD pmf right hand state proportion: 
0.6306
TI left hand state proportion: 
0.1201
TI middle hand state proportion: 
0.3335
TI right hand state proportion: 
0.5464


#### Computing corrected charging step bound state dG

This is simply computing the dG of each state then combining them in a weighted sum, weighted by the microstates in the GaMD pmf.

In [22]:
corr_crg_dGs = a1_prop * st1boost_dGs + a2_prop * st2boost_dGs + a3_prop * st3boost_dGs

## Comparing GaMD with TI, unbound state

In [23]:
os.chdir("../../gamd_pmfs/VL-R66G")
v164_pmf_ub = get_pmf_2d(
    "./pmf-c2-V29c1c2_conv_300ns.dat.xvg"
)

geom_dvdls_crg_ub["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_crg_ub, "V249_chi1", "V249_chi2", v164_pmf_ub, 10)


In [25]:
fig = plot_pmf_with_TI_2d_fig(v164_pmf_ub, geom_dvdls_crg_ub[geom_dvdls_crg_ub["in_pmf_cont"]], "V249_chi1", "V249_chi2")
fig.update_xaxes(
    title="VL-V29 chi1"
).update_yaxes(
    title="VL-V29 chi2"
).update_layout(width=1000, height=700)
fig.show(renderer="svg")

### Checking sampling of the three microstates

The leftmost and rightmost microstates have insufficient sampling and need boosted TI lambda production sampling.

In [29]:
st1_ub = geom_dvdls_crg_ub[
    (geom_dvdls_crg_ub["in_pmf_cont"]) &
    (geom_dvdls_crg_ub["V249_chi1"] > -100) & 
    (geom_dvdls_crg_ub["V249_chi1"] < -25) 
]

st2_ub = geom_dvdls_crg_ub[
    (geom_dvdls_crg_ub["in_pmf_cont"]) &
    (geom_dvdls_crg_ub["V249_chi1"] > 0) & 
    (geom_dvdls_crg_ub["V249_chi1"] < 100)
]

st3_ub = geom_dvdls_crg_ub[
    (geom_dvdls_crg_ub["in_pmf_cont"]) &
    ((geom_dvdls_crg_ub["V249_chi1"] > 125) | 
    (geom_dvdls_crg_ub["V249_chi1"] < -100))
]

print(st1_ub.groupby("Lambda").count()["weight_dvdl"])
print(st2_ub.groupby("Lambda").count()["weight_dvdl"])
print(st3_ub.groupby("Lambda").count()["weight_dvdl"])

Lambda
1     133
4       5
7      16
8       9
9     179
10    164
12     56
Name: weight_dvdl, dtype: int64
Lambda
1     411
2     547
3     822
4     700
5     882
6     880
7     749
8     874
9     721
10    723
11    866
12    650
Name: weight_dvdl, dtype: int64
Lambda
1     416
2     410
3     138
4     246
5      75
6      60
7     144
8      30
9      29
10     33
11     61
12    221
Name: weight_dvdl, dtype: int64


### Ingesting boosted TI lambda production sampling on the leftmost and rightmost states

In [31]:
os.chdir("../../TI_data/VL-R66G")
sc_dvdls_crgUb_boost = pd.read_csv("R66G_crg_boost_ub.csv")

In [32]:
# sc_dvdls_crgUb_boost = pd.concat([geom_dvdls_crg_ub, sc_dvdls_crg_ub_more_st1, sc_dvdls_crg_ub_more_st1_2, sc_dvdls_crg_ub_more_st3])
sc_dvdls_crgUb_boost["in_pmf_cont"] = rot_in_pmf_cont(sc_dvdls_crgUb_boost, "V249_chi1", "V249_chi2", v164_pmf_ub, 10)


### Checking that the sampling is now sufficient

In [34]:
st1boost_ub = sc_dvdls_crgUb_boost[
    (sc_dvdls_crgUb_boost["in_pmf_cont"]) &
    (sc_dvdls_crgUb_boost["V249_chi1"] > -100) & 
    (sc_dvdls_crgUb_boost["V249_chi1"] < -25) 
]

st2boost_ub = sc_dvdls_crgUb_boost[
    (sc_dvdls_crgUb_boost["in_pmf_cont"]) &
    (sc_dvdls_crgUb_boost["V249_chi1"] > 0) & 
    (sc_dvdls_crgUb_boost["V249_chi1"] < 100)
]

st3boost_ub = sc_dvdls_crgUb_boost[
    (sc_dvdls_crgUb_boost["in_pmf_cont"]) &
    (sc_dvdls_crgUb_boost["V249_chi1"] > 125) | 
    (sc_dvdls_crgUb_boost["V249_chi1"] < -150)
]

print(st1boost_ub.groupby("Lambda").count()["weight_dvdl"])
print(st2boost_ub.groupby("Lambda").count()["weight_dvdl"])
print(st3boost_ub.groupby("Lambda").count()["weight_dvdl"])


Lambda
1     1272
2     1319
3      197
4      201
5      138
6      500
7      315
8      268
9     1447
10    1781
11    1563
12    1084
Name: weight_dvdl, dtype: int64
Lambda
1      989
2     1616
3     2780
4     2383
5     3024
6     2577
7     2234
8     2884
9     1301
10    1107
11    1426
12    1620
Name: weight_dvdl, dtype: int64
Lambda
1     1490
2      750
3      807
4     1224
5      658
6      642
7     1198
8      612
9      952
10     782
11     638
12     958
Name: weight_dvdl, dtype: int64


### Comparing TI sampling of the three microstates to the GaMD free energy profile areas

In [38]:

a1ub = v164_pmf_ub[(v164_pmf_ub["X"] > -120) & (v164_pmf_ub["X"] < -25)]["pop"].sum()
a2ub = v164_pmf_ub[(v164_pmf_ub["X"] > 0) & (v164_pmf_ub["X"] < 120)]["pop"].sum()
a3ub = v164_pmf_ub[(v164_pmf_ub["X"] > 120) | (v164_pmf_ub["X"] < -150)]["pop"].sum()

a1_prop_ub = a1ub/(a1ub + a2ub + a3ub)
a2_prop_ub = a2ub/(a1ub + a2ub + a3ub)
a3_prop_ub = a3ub/(a1ub + a2ub + a3ub)

print("GaMD pmf left hand state proportion: ")
print(round(a1_prop_ub, 4))
print("GaMD pmf middle state proportion: ")
print(round(a2_prop_ub, 4))
print("GaMD pmf right hand state proportion: ")
print(round(a3_prop_ub, 4))


ti1_ub = len(st1_ub)
ti2_ub = len(st2_ub)
ti3_ub = len(st3_ub)

print()
print("TI original, unboosted proportions")
print("TI left hand state proportion: ")
print(round(ti1_ub/(ti1_ub + ti2_ub + ti3_ub), 4))
print("TI middle hand state proportion: ")
print(round(ti2_ub/(ti1_ub + ti2_ub + ti3_ub), 4))
print("TI right hand state proportion: ")
print(round(ti3_ub/(ti1_ub + ti2_ub + ti3_ub), 4))


GaMD pmf left hand state proportion: 
0.0379
GaMD pmf middle state proportion: 
0.2124
GaMD pmf right hand state proportion: 
0.7497

TI original, unboosted proportions
TI left hand state proportion: 
0.05
TI middle hand state proportion: 
0.7844
TI right hand state proportion: 
0.1656


#### Recomputing unbound charging step dG and corrected charging step ddG

In [41]:
st1boost_ub_dGs = np.array(bootstrap_df(st1boost_ub, 1000, 200))
st2boost_ub_dGs = np.array(bootstrap_df(st2boost_ub, 1000, 200))
st3boost_ub_dGs = np.array(bootstrap_df(st3boost_ub, 1000, 200))

In [43]:
corr_crg_ub_dGs = a1_prop_ub * st1boost_ub_dGs + a2_prop_ub * st2boost_ub_dGs + a3_prop_ub * st3boost_ub_dGs

#### Charging step correction results

With the correction based on the VL-V29 chi1/chi2 rotamer sampling, there is slight change in the charging step ddG (~0.3 kcal/mol increase)

Next, we will incorporate the corrections from the vdw step (found in the `R66Gvdw_example.ipynb` notebook).

In [44]:
print("Values after crg step correction, no vdw step correction")

corr_ddGs = print_summary_crg(corr_crg_dGs, corr_crg_ub_dGs, orig_vdw_dGs, orig_vdw_ub_dGs)

Values after crg step correction, no vdw step correction
Charging step ddG: -0.5294 +/- 0.6354
Lower CI: -1.1129
Upper CI: 0.1061

Vdw step ddG: 0.1667 +/- 0.607
Lower CI: -0.4404
Upper CI: 0.75

Total ddG: -0.3627 +/- 0.9273
Lower CI: -1.29
Upper CI: 0.484


#### Original results for comparison 

In [45]:
total_ddGs = print_summary_crg(orig_crg_dGs, orig_crg_ub_dGs, orig_vdw_dGs, orig_vdw_ub_dGs)


Charging step ddG: -0.8569 +/- 0.8902
Lower CI: -1.7064
Upper CI: 0.0334

Vdw step ddG: 0.1667 +/- 0.607
Lower CI: -0.4404
Upper CI: 0.75

Total ddG: -0.6902 +/- 1.0979
Lower CI: -1.7723
Upper CI: 0.4077


#### Incorporating vdw step correction (bound state)

We will concisely reproduce the correction here, but for more detail please see the `R66Gvdw_example.ipynb` notebook. 

In [60]:
# importing GaMD pmf for bound state VL-F71 chi2, VL-D28 chi1 (most influential rotamers according to vdw step RF model)
os.chdir("../../gamd_pmfs/VL-R66G")
f71c2_d28c1 = get_pmf_2d(
    "pmf-c2-F71c2_D28c1_bound.dat.xvg"
)
f71c2_d28c1.loc[f71c2_d28c1["Y"] < 0, "Y"] += 360

# importing GaMD pmf for unbound state VL-F71 chi2, VL-D28 chi1 (most influential rotamers according to vdw step RF model)
f71c2_d28c1_ub = get_pmf_2d(
    "pmf-c2-F71c2_D28c1_unbound.dat.xvg"
)
f71c2_d28c1_ub.loc[f71c2_d28c1_ub["Y"] < 0, "Y"] += 360

# shifting coordinates for simplicity (-180, 180) --> (0, 360) 
f71c2_d28c1.loc[f71c2_d28c1["Y"] < 0, "Y"] += 360
f71c2_d28c1_ub.loc[f71c2_d28c1_ub["Y"] < 0, "Y"] += 360
geom_dvdls_vdw.loc[geom_dvdls_vdw["D383_chi1"] < 0, "D383_chi1"] += 360

# filtering TI sampling to match GaMD pmf
geom_dvdls_vdw["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_vdw, "F426_chi2", "D383_chi1", f71c2_d28c1, 10)

# checking sufficient sampling, computing corrected dG via bootstrapping
f71c1_gr_0 = geom_dvdls_vdw[(geom_dvdls_vdw["in_pmf_cont"]) & (geom_dvdls_vdw["F426_chi2"] > 0)]
print(f71c1_gr_0.groupby("Lambda").count()["weight_dvdl"])
corr_vdw_dGs = bootstrap_df(f71c1_gr_0, 1000, 200)

Lambda
1     362
2     343
3     335
4     331
5     348
6     332
7     349
8     382
9     361
10    363
11    386
12    383
Name: weight_dvdl, dtype: int64


#### Incorporating vdw step correction (unbound state)

We will concisely reproduce the correction here, but for more detail please see the `R66Gvdw_example.ipynb` notebook. 

In [61]:
# shifting coordinates for simplicity (-180, 180) --> (0, 360)
geom_dvdls_vdw_ub.loc[geom_dvdls_vdw_ub["D248_chi1"] < 0, "D248_chi1"] += 360

# filtering TI sampling to match GaMD pmf 
geom_dvdls_vdw_ub["in_pmf_cont"] = rot_in_pmf_cont(geom_dvdls_vdw_ub, "F291_chi2", "D248_chi1", f71c2_d28c1_ub, 10)

# computing area of both microstates of pmf and their relative proportions
a1ub = f71c2_d28c1_ub[f71c2_d28c1_ub["X"] < 0]["pop"].sum()
a2ub = f71c2_d28c1_ub[(f71c2_d28c1_ub["X"] > 0)]["pop"].sum()
a1_prop_ub = a1ub/(a1ub + a2ub)
a2_prop_ub = a2ub/(a1ub + a2ub)


# checking sampling of both microstates 
st1ub = geom_dvdls_vdw_ub[(geom_dvdls_vdw_ub["in_pmf_cont"]) & (geom_dvdls_vdw_ub["F291_chi2"] < 0)]
st2ub = geom_dvdls_vdw_ub[
    (geom_dvdls_vdw_ub["in_pmf_cont"]) & 
    (geom_dvdls_vdw_ub["F291_chi2"] > 0) 
]
print(st1ub.groupby("Lambda").count()["weight_dvdl"])
print(st2ub.groupby("Lambda").count()["weight_dvdl"])

# computing dG values for each microstate
st1_ub_dGs = np.array(bootstrap_df(st1ub, 1000, 200))
st2_ub_dGs = np.array(bootstrap_df(st2ub, 1000, 200))

# computing correction based on GaMD pmf proportions
corr_vdw_ub_dGs = a1_prop_ub * st1_ub_dGs + a2_prop_ub * st2_ub_dGs

Lambda
1     214
2     228
3     258
4     282
5     253
6     290
7     218
8     255
9     316
10    321
11    326
12    334
Name: weight_dvdl, dtype: int64
Lambda
1     265
2     368
3     267
4     278
5     235
6     251
7     274
8     281
9     264
10    290
11    260
12    283
Name: weight_dvdl, dtype: int64


#### Charging + vdw step correction results

With the charging step correction based on the VL-V29 chi1/chi2 rotamer sampling, there is slight change in the charging step ddG (\~0.3 kcal/mol increase)
With the vdw step correction based on VL-F71 chi2 and VL-D28 chi1 rotamer sampling, there is a significant change in vdw step ddG (\~1 kcal/mol increase)

In [62]:
print("Values after corrections to both crg step and vdw step")

corr_ddGs_fin = print_summary_crg(corr_crg_dGs, corr_crg_ub_dGs, corr_vdw_dGs, corr_vdw_ub_dGs)

corr_err = abs(empirical_value - np.mean(corr_ddGs_fin))
print()
print(f"\nEmpirical value: {empirical_value} kcal/mol")
print(f"Error after corrections: {round(corr_err, 4)} kcal/mol")

Values after corrections to both crg step and vdw step
Charging step ddG: -0.5294 +/- 0.6354
Lower CI: -1.1129
Upper CI: 0.1061

Vdw step ddG: 1.3165 +/- 0.5403
Lower CI: 0.7762
Upper CI: 1.8341

Total ddG: 0.7871 +/- 0.8721
Lower CI: 0.0333
Upper CI: 1.6592


Empirical value: 0.22 kcal/mol
Error after corrections: 0.5671 kcal/mol


#### Original values for comparison

In [59]:
total_ddGs = print_summary_crg(orig_crg_dGs, orig_crg_ub_dGs, orig_vdw_dGs, orig_vdw_ub_dGs)
print(f"\nEmpirical value: {empirical_value} kcal/mol")
print(f"Original error: {round(orig_error, 4)} kcal/mol")

Charging step ddG: -0.8569 +/- 0.8902
Lower CI: -1.7064
Upper CI: 0.0334

Vdw step ddG: 0.1667 +/- 0.607
Lower CI: -0.4404
Upper CI: 0.75

Total ddG: -0.6902 +/- 1.0979
Lower CI: -1.7723
Upper CI: 0.4077

Empirical value: 0.22 kcal/mol
Original error: 0.9102 kcal/mol
